# WiDS 2026 Wildfire Survival — 5km Late-Gate Ensemble

In [1]:

import os, math, warnings, random
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # "full" for Kaggle submission, "fast" for quick smoke-test
OUTPUT_PATH = "/kaggle/working/submission.csv" if os.path.isdir("/kaggle/working") else "submission.csv"
SEEDS_FULL = (17, 43, 91, 137, 211, 431, 733, 2027)
SEEDS_FAST = (17, 137, 733)
SEEDS = SEEDS_FULL if RUN_MODE == "full" else SEEDS_FAST
RANDOM_SEARCH_N = 6000 if RUN_MODE == "full" else 800

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge, BayesianRidge, HuberRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.isotonic import IsotonicRegression

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    HAS_CAT = True
except Exception:
    HAS_CAT = False

def find_data_dir():
    direct = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide_globaldathon26",
        "/kaggle/input",
        ".",
        "/mnt/data",
    ]
    need = {"train.csv", "test.csv", "sample_submission.csv"}
    for d in direct:
        if os.path.isdir(d) and need.issubset(set(os.listdir(d))):
            return d
    for root in ["/kaggle/input", "/mnt/data", "."]:
        if not os.path.isdir(root):
            continue
        for path, _, files in os.walk(root):
            if need.issubset(set(files)):
                return path
    raise FileNotFoundError("train.csv, test.csv, sample_submission.csv not found.")

DATA_DIR = find_data_dir()
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

ID = "event_id"
TARGET_TIME = "time_to_hit_hours"
TARGET_EVENT = "event"
HORIZONS = [12, 24, 48, 72]
P_COLS = ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]

time_values = train[TARGET_TIME].to_numpy(float)
event_values = train[TARGET_EVENT].to_numpy(int)

def enforce_monotone(arr):
    arr = np.clip(np.asarray(arr, dtype=float), 0.0, 1.0)
    for j in range(1, arr.shape[1]):
        arr[:, j] = np.maximum(arr[:, j], arr[:, j - 1])
    return arr

def c_index(time, event, risk):
    conc = 0.0
    comp = 0.0
    n = len(time)
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1.0
            if risk[i] > risk[j]:
                conc += 1.0
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def brier_at(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    y = ((event == 1) & (time <= horizon)).astype(float)[valid]
    p = np.clip(prob[valid], 0.0, 1.0)
    return float(np.mean((p - y) ** 2))

def hybrid_score(pred):
    pred = enforce_monotone(pred)
    risk = 0.3 * pred[:, 1] + 0.4 * pred[:, 2] + 0.3 * pred[:, 3]
    ci = c_index(time_values, event_values, risk)
    b24 = brier_at(time_values, event_values, pred[:, 1], 24)
    b48 = brier_at(time_values, event_values, pred[:, 2], 48)
    b72 = brier_at(time_values, event_values, pred[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * ci + 0.7 * (1.0 - wb), ci, wb, b24, b48, b72

def make_features(df):
    r = df.copy()
    dist = np.maximum(r["dist_min_ci_0_5h"].to_numpy(float), 1.0)
    area = np.maximum(r["area_first_ha"].to_numpy(float), 0.0)
    radius_m = np.sqrt(area * 10000.0 / np.pi)
    close = np.maximum(r["closing_speed_m_per_h"].to_numpy(float), 0.0)
    radial = np.maximum(r["radial_growth_rate_m_per_h"].to_numpy(float), 0.0)
    eff_speed = close + radial
    perim = r["num_perimeters_0_5h"].to_numpy(float)
    low = r["low_temporal_resolution_0_5h"].to_numpy(float)
    align = r["alignment_abs"].to_numpy(float)

    r["dist_km"] = dist / 1000.0
    r["log_dist"] = np.log1p(dist)
    r["inv_dist_km"] = 1.0 / (dist / 1000.0 + 0.10)
    r["dist_to_5k_m"] = dist - 5000.0
    r["inside5k_m"] = np.maximum(5000.0 - dist, 0.0)
    r["outside5k_m"] = np.maximum(dist - 5000.0, 0.0)
    r["near5k_band"] = ((dist >= 4500.0) & (dist <= 6500.0)).astype(float)
    r["under5k"] = (dist < 5000.0).astype(float)

    r["fire_radius_m"] = radius_m
    r["fire_radius_km"] = radius_m / 1000.0
    r["radius_to_dist"] = radius_m / dist
    r["radius_to_gap"] = radius_m / (np.abs(dist - 5000.0) + 100.0)

    r["eff_closing_speed"] = eff_speed
    r["log_eff_speed"] = np.log1p(eff_speed)
    r["eta_center_h"] = dist / (eff_speed + 1e-3)
    r["eta_gap_h"] = np.maximum(dist - 5000.0, 0.0) / (eff_speed + 1e-3)
    r["log_eta_center"] = np.log1p(np.clip(r["eta_center_h"], 0, 1e6))
    r["log_eta_gap"] = np.log1p(np.clip(r["eta_gap_h"], 0, 1e6))

    r["log_area"] = np.log1p(area)
    r["sqrt_area"] = np.sqrt(area)
    r["small_area"] = (area < 30.0).astype(float)
    r["tiny_area"] = (area < 10.0).astype(float)
    r["large_area"] = (area > 500.0).astype(float)
    r["giant_area"] = (area > 1500.0).astype(float)

    r["has_motion"] = (perim > 1.0).astype(float)
    r["multi_perim"] = (perim >= 2.0).astype(float)
    r["many_perim"] = (perim >= 5.0).astype(float)
    r["very_many_perim"] = (perim >= 10.0).astype(float)
    r["perim_log"] = np.log1p(perim)
    r["has_growth"] = (eff_speed > 1e-6).astype(float)

    r["lowres_small"] = low * r["small_area"]
    r["lowres_tiny"] = low * r["tiny_area"]
    r["lowres_large"] = low * r["large_area"]
    r["lowres_oneperim"] = low * (perim <= 1.0).astype(float)

    hr = r["event_start_hour"].to_numpy(float)
    mo = r["event_start_month"].to_numpy(float)
    dow = r["event_start_dayofweek"].to_numpy(float)
    r["hour_sin"] = np.sin(2.0 * np.pi * hr / 24.0)
    r["hour_cos"] = np.cos(2.0 * np.pi * hr / 24.0)
    r["month_sin"] = np.sin(2.0 * np.pi * mo / 12.0)
    r["month_cos"] = np.cos(2.0 * np.pi * mo / 12.0)
    r["dow_sin"] = np.sin(2.0 * np.pi * dow / 7.0)
    r["dow_cos"] = np.cos(2.0 * np.pi * dow / 7.0)
    r["night_or_late"] = ((hr <= 5.0) | (hr >= 22.0)).astype(float)
    r["evening"] = ((hr >= 18.0) & (hr <= 22.0)).astype(float)
    r["midday"] = ((hr >= 10.0) & (hr <= 17.0)).astype(float)

    r["align_motion"] = align * r["has_motion"]
    r["align_inv_dist"] = align * r["inv_dist_km"]
    r["perim_area"] = r["perim_log"] * r["log_area"]
    r["motion_area"] = r["has_motion"] * r["log_area"]
    r["growth_area"] = r["log_eff_speed"] * r["log_area"]
    r["area_dist_ratio"] = r["log_area"] * r["inv_dist_km"]
    r["late_slow_signature"] = r["lowres_oneperim"] * (0.8 * r["small_area"] + 0.5 * r["tiny_area"] + 0.25 * r["midday"])

    r["physics_fast_score"] = (
        (5000.0 - dist) / 1000.0
        + 0.28 * r["log_area"]
        + 0.65 * r["multi_perim"]
        + 0.70 * r["many_perim"]
        + 0.35 * align
        + 0.35 * r["has_growth"]
        - 0.85 * low
        - 0.55 * r["small_area"]
        - 0.40 * r["tiny_area"]
        - 0.20 * r["night_or_late"]
    )

    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return r

train_fe = make_features(train)
test_fe = make_features(test)
drop_cols = [ID, TARGET_TIME, TARGET_EVENT]
feature_cols = [c for c in train_fe.columns if c not in drop_cols]
X_all = train_fe[feature_cols]
X_test = test_fe[feature_cols]

positive_idx = np.where(event_values == 1)[0]
pos_time = time_values[positive_idx]
X_pos = X_all.iloc[positive_idx].reset_index(drop=True)

def event_gate_and_floor(df):
    dist = df["dist_min_ci_0_5h"].to_numpy(float)
    gap = np.maximum(dist - 5000.0, 0.0)
    moving = (df["num_perimeters_0_5h"].to_numpy(float) >= 2.0).astype(float)
    align = df["alignment_abs"].to_numpy(float)
    growth = (
        np.maximum(df["radial_growth_rate_m_per_h"].to_numpy(float), 0.0)
        + np.maximum(df["closing_speed_m_per_h"].to_numpy(float), 0.0)
    )
    far_floor = 0.008 + 0.018 * np.exp(-gap / 16000.0)
    far_floor += 0.006 * moving * np.exp(-gap / 25000.0)
    far_floor += 0.004 * (align > 0.7).astype(float) * np.exp(-gap / 30000.0)
    far_floor += 0.004 * (growth > 1.0).astype(float) * np.exp(-gap / 30000.0)
    gate = np.where(dist < 5000.0, 0.993, np.clip(far_floor, 0.006, 0.055))
    floor_by_h = np.column_stack([
        np.clip(far_floor * 0.70, 0.006, 0.045),
        np.clip(far_floor * 0.92, 0.008, 0.055),
        np.clip(far_floor * 1.05, 0.010, 0.065),
    ])
    return gate, floor_by_h

gate_train, floor_train = event_gate_and_floor(train)
gate_test, floor_test = event_gate_and_floor(test)

def physics_conditional(df):
    dist = df["dist_min_ci_0_5h"].to_numpy(float)
    area = np.maximum(df["area_first_ha"].to_numpy(float), 0.0)
    perim = df["num_perimeters_0_5h"].to_numpy(float)
    low = df["low_temporal_resolution_0_5h"].to_numpy(float)
    align = df["alignment_abs"].to_numpy(float)
    hr = df["event_start_hour"].to_numpy(float)
    growth = (
        np.maximum(df["radial_growth_rate_m_per_h"].to_numpy(float), 0.0)
        + np.maximum(df["closing_speed_m_per_h"].to_numpy(float), 0.0)
    )

    score = np.zeros(len(df), dtype=float)
    score += 1.08 * (perim >= 2.0)
    score += 0.92 * (perim >= 3.0)
    score += 0.90 * (perim >= 5.0)
    score += 0.30 * np.log1p(area)
    score += 0.55 * (align > 0.50)
    score += 0.35 * (align > 0.85)
    score += 0.45 * (growth > 1.0)
    score += 0.26 * (dist < 1200.0)
    score += 0.10 * (dist < 700.0)
    score -= 0.11 * (dist > 3200.0)
    score -= 0.18 * (dist > 4500.0)
    score -= 1.05 * low
    score -= 0.58 * (area < 30.0)
    score -= 0.42 * (area < 10.0)
    score -= 0.12 * ((hr <= 5.0) | (hr >= 23.0))
    score += 0.18 * ((hr >= 18.0) & (hr <= 22.0))
    score -= 0.20 * ((low == 1.0) & (perim <= 1.0) & (area < 20.0) & (hr >= 14.0) & (hr <= 18.0))

    p12 = expit(score - 0.28)
    p24 = expit(score + 1.02)
    p48 = expit(score + 1.88)
    return np.column_stack([p12, p24, p48])

def group_names(df):
    perim = df["num_perimeters_0_5h"].to_numpy(float)
    low = df["low_temporal_resolution_0_5h"].to_numpy(float)
    area = df["area_first_ha"].to_numpy(float)
    return np.select(
        [
            (perim >= 2.0) & (low == 0.0),
            (low == 1.0) & (area < 30.0),
            (low == 1.0) & (area >= 30.0),
            (perim >= 2.0),
        ],
        ["multi_good", "low_small", "low_not_small", "multi_other"],
        default="other",
    )

pos_df = train.iloc[positive_idx].copy().reset_index(drop=True)
pos_groups = group_names(pos_df)
global_pos_rates = np.array([(pos_time <= h).mean() for h in [12, 24, 48]], dtype=float)

def group_candidate():
    alpha = 5.0
    rates = {}
    for g in np.unique(pos_groups):
        m = pos_groups == g
        n = float(m.sum())
        counts = np.array([np.sum(pos_time[m] <= h) for h in [12, 24, 48]], dtype=float)
        rates[g] = (counts + alpha * global_pos_rates) / (n + alpha)

    pos_oof = np.zeros((len(pos_time), 3), dtype=float)
    for i, g in enumerate(pos_groups):
        m = (pos_groups == g)
        m[i] = False
        n = float(m.sum())
        if n <= 0:
            pos_oof[i] = global_pos_rates
        else:
            counts = np.array([np.sum(pos_time[m] <= h) for h in [12, 24, 48]], dtype=float)
            pos_oof[i] = (counts + alpha * global_pos_rates) / (n + alpha)

    test_cond = np.vstack([rates.get(g, global_pos_rates) for g in group_names(test)])
    return pos_oof, test_cond

def wrap_conditional_candidate(pos_oof, test_cond, name):
    train_pred = np.zeros((len(train), 4), dtype=float)
    train_pred[:, :3] = floor_train
    train_pred[positive_idx, :3] = (
        gate_train[positive_idx, None] * np.clip(pos_oof, 0, 1)
        + (1.0 - gate_train[positive_idx, None]) * floor_train[positive_idx]
    )
    train_pred[:, 3] = 1.0

    test_pred = np.zeros((len(test), 4), dtype=float)
    test_pred[:, :3] = np.where(
        (test["dist_min_ci_0_5h"].to_numpy(float) < 5000.0)[:, None],
        gate_test[:, None] * np.clip(test_cond, 0, 1) + (1.0 - gate_test[:, None]) * floor_test,
        floor_test,
    )
    test_pred[:, 3] = 1.0
    return name, enforce_monotone(train_pred), enforce_monotone(test_pred)

def pos_cv_splits():
    bins = np.digitize(pos_time, [12.0, 24.0, 48.0])
    counts = np.bincount(bins)
    min_count = int(counts[counts > 0].min())
    n_splits = max(2, min(3, min_count))
    for seed in SEEDS:
        yield StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X_pos, bins)

def pos_classifier_candidate(builder, name):
    y_mat = np.column_stack([(pos_time <= h).astype(int) for h in [12, 24, 48]])
    oof = np.zeros((len(pos_time), 3), dtype=float)
    cnt = np.zeros(len(pos_time), dtype=float)
    test_acc = np.zeros((len(test), 3), dtype=float)
    xt = X_test[feature_cols]

    for split_iter in pos_cv_splits():
        for tr_idx, va_idx in split_iter:
            cnt[va_idx] += 1.0
            for j, h in enumerate([12, 24, 48]):
                y = y_mat[:, j]
                if len(np.unique(y[tr_idx])) < 2:
                    oof[va_idx, j] += y[tr_idx].mean()
                else:
                    model = builder(j, h)
                    model.fit(X_pos.iloc[tr_idx], y[tr_idx])
                    oof[va_idx, j] += model.predict_proba(X_pos.iloc[va_idx])[:, 1]

    oof /= cnt[:, None]

    for seed in SEEDS:
        for j, h in enumerate([12, 24, 48]):
            y = y_mat[:, j]
            if len(np.unique(y)) < 2:
                test_acc[:, j] += y.mean()
            else:
                model = builder(j, h, seed=seed)
                model.fit(X_pos, y)
                test_acc[:, j] += model.predict_proba(xt)[:, 1]
    test_cond = test_acc / len(SEEDS)
    return wrap_conditional_candidate(oof, test_cond, name)

def pos_regression_candidate(builder, name):
    y_log = np.log1p(pos_time)
    oof = np.zeros((len(pos_time), 3), dtype=float)
    cnt = np.zeros(len(pos_time), dtype=float)
    test_acc = np.zeros((len(test), 3), dtype=float)

    for split_iter in pos_cv_splits():
        for tr_idx, va_idx in split_iter:
            model = builder()
            model.fit(X_pos.iloc[tr_idx], y_log[tr_idx])
            mu_va = model.predict(X_pos.iloc[va_idx])
            mu_tr = model.predict(X_pos.iloc[tr_idx])
            resid = y_log[tr_idx] - mu_tr
            for j, h in enumerate([12, 24, 48]):
                z = np.log1p(h) - mu_va
                oof[va_idx, j] += np.mean(resid[None, :] <= z[:, None], axis=1)
            cnt[va_idx] += 1.0

    oof /= cnt[:, None]

    for seed in SEEDS:
        model = builder(seed=seed)
        model.fit(X_pos, y_log)
        mu_test = model.predict(X_test[feature_cols])
        mu_train = model.predict(X_pos)
        resid = y_log - mu_train
        for j, h in enumerate([12, 24, 48]):
            z = np.log1p(h) - mu_test
            test_acc[:, j] += np.mean(resid[None, :] <= z[:, None], axis=1)
    test_cond = test_acc / len(SEEDS)
    return wrap_conditional_candidate(oof, test_cond, name)

def all_horizon_classifier_candidate(builder, name):
    train_pred = np.zeros((len(train), 4), dtype=float)
    test_pred = np.zeros((len(test), 4), dtype=float)
    full_fill = np.zeros((len(train), 3), dtype=float)

    for j, h in enumerate([12, 24, 48]):
        unknown = (event_values == 0) & (time_values < h)
        valid = ~unknown
        y = ((event_values == 1) & (time_values <= h)).astype(int)
        idx = np.where(valid)[0]
        yv = y[idx]
        min_class = np.bincount(yv).min()
        n_splits = max(2, min(5, int(min_class)))
        oof = np.zeros(len(train), dtype=float)
        cnt = np.zeros(len(train), dtype=float)
        test_acc = np.zeros(len(test), dtype=float)
        fill_acc = np.zeros(len(train), dtype=float)

        for seed in SEEDS:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            for tr_sub, va_sub in cv.split(X_all.iloc[idx], yv):
                tr_idx = idx[tr_sub]
                va_idx = idx[va_sub]
                model = builder(j, h, seed=seed)
                model.fit(X_all.iloc[tr_idx], y[tr_idx])
                oof[va_idx] += model.predict_proba(X_all.iloc[va_idx])[:, 1]
                cnt[va_idx] += 1.0

            model = builder(j, h, seed=seed)
            model.fit(X_all.iloc[idx], yv)
            test_acc += model.predict_proba(X_test)[:, 1]
            fill_acc += model.predict_proba(X_all)[:, 1]

        m = cnt > 0
        oof[m] /= cnt[m]
        fill_acc /= len(SEEDS)
        oof[~m] = fill_acc[~m]
        test_acc /= len(SEEDS)
        train_pred[:, j] = oof
        test_pred[:, j] = test_acc

    # Blend all-data learner with the deterministic 5km event gate to avoid leakage-prone far positives.
    train_gate = np.zeros((len(train), 4), dtype=float)
    train_gate[:, :3] = floor_train
    train_gate[positive_idx, :3] = train_pred[positive_idx, :3]
    train_gate[:, 3] = 1.0

    test_gate = np.zeros((len(test), 4), dtype=float)
    near_test = (test["dist_min_ci_0_5h"].to_numpy(float) < 5000.0)
    test_gate[:, :3] = np.where(near_test[:, None], test_pred[:, :3], floor_test)
    test_gate[:, 3] = 1.0

    train_pred[:, 3] = 1.0
    test_pred[:, 3] = 1.0
    train_mix = 0.70 * train_gate + 0.30 * train_pred
    test_mix = 0.70 * test_gate + 0.30 * test_pred
    return name, enforce_monotone(train_mix), enforce_monotone(test_mix)

def build_logit(j=None, h=None, seed=17, C=0.18, balanced=False):
    return Pipeline([
        ("scale", RobustScaler()),
        ("lr", LogisticRegression(
            C=C,
            solver="liblinear",
            class_weight="balanced" if balanced else None,
            random_state=seed,
            max_iter=2000,
        )),
    ])

def build_knn(j=None, h=None, seed=17):
    return Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=9, weights="distance")),
    ])

def build_et_cls(j=None, h=None, seed=17):
    return ExtraTreesClassifier(
        n_estimators=90 if RUN_MODE == "full" else 40,
        max_depth=3,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        random_state=seed,
        n_jobs=1,
    )

def build_gb_cls(j=None, h=None, seed=17):
    return GradientBoostingClassifier(
        n_estimators=70 if RUN_MODE == "full" else 35,
        learning_rate=0.045,
        max_depth=1,
        min_samples_leaf=5,
        subsample=0.85,
        random_state=seed,
    )

def build_ridge(seed=17):
    return Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=8.0))])

def build_bayes(seed=17):
    return Pipeline([("scale", StandardScaler()), ("br", BayesianRidge())])

def build_huber(seed=17):
    return Pipeline([("scale", StandardScaler()), ("huber", HuberRegressor(alpha=0.7, epsilon=1.25, max_iter=1000))])

def build_et_reg(seed=17):
    return ExtraTreesRegressor(
        n_estimators=110 if RUN_MODE == "full" else 45,
        max_depth=3,
        min_samples_leaf=4,
        random_state=seed,
        n_jobs=1,
    )

candidate_train = []
candidate_test = []
candidate_names = []

# Deterministic/domain candidates
phys_cond_train_pos = physics_conditional(pos_df)
phys_cond_test = physics_conditional(test)
name, trp, tep = wrap_conditional_candidate(phys_cond_train_pos, phys_cond_test, "physics_5km_timing")
candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)

grp_oof, grp_test = group_candidate()
name, trp, tep = wrap_conditional_candidate(grp_oof, grp_test, "loo_group_km_timing")
candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)

# Positive-only timing candidates
for nm, builder in [
    ("pos_ridge_residual_cdf", build_ridge),
    ("pos_bayes_residual_cdf", build_bayes),
    ("pos_huber_residual_cdf", build_huber),
    ("pos_extratrees_residual_cdf", build_et_reg),
]:
    try:
        name, trp, tep = pos_regression_candidate(builder, nm)
        candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)
    except Exception as e:
        print("skip", nm, str(e)[:120])

for nm, builder in [
    ("pos_logit", lambda j, h, seed=17: build_logit(j, h, seed, C=0.16, balanced=False)),
    ("pos_logit_balanced", lambda j, h, seed=17: build_logit(j, h, seed, C=0.35, balanced=True)),
    ("pos_knn", build_knn),
    ("pos_extratrees_cls", build_et_cls),
    ("pos_gb_stump", build_gb_cls),
]:
    try:
        name, trp, tep = pos_classifier_candidate(builder, nm)
        candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)
    except Exception as e:
        print("skip", nm, str(e)[:120])

if HAS_LGB:
    def build_lgb_cls(j=None, h=None, seed=17):
        return lgb.LGBMClassifier(
            objective="binary",
            n_estimators=90 if RUN_MODE == "full" else 45,
            learning_rate=0.035,
            num_leaves=4,
            max_depth=2,
            min_child_samples=6,
            subsample=0.85,
            colsample_bytree=0.70,
            reg_alpha=1.0,
            reg_lambda=4.0,
            random_state=seed,
            n_jobs=1,
            verbosity=-1,
        )
    def build_lgb_reg(seed=17):
        return lgb.LGBMRegressor(
            objective="regression",
            n_estimators=95 if RUN_MODE == "full" else 45,
            learning_rate=0.035,
            num_leaves=4,
            max_depth=2,
            min_child_samples=6,
            subsample=0.85,
            colsample_bytree=0.70,
            reg_alpha=1.0,
            reg_lambda=4.0,
            random_state=seed,
            n_jobs=1,
            verbosity=-1,
        )
    for nm, fn, is_reg in [
        ("pos_lgb_residual_cdf", build_lgb_reg, True),
        ("pos_lgb_cls", build_lgb_cls, False),
    ]:
        try:
            if is_reg:
                name, trp, tep = pos_regression_candidate(fn, nm)
            else:
                name, trp, tep = pos_classifier_candidate(fn, nm)
            candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)
        except Exception as e:
            print("skip", nm, str(e)[:120])

if HAS_CAT:
    def build_cat_cls(j=None, h=None, seed=17):
        return CatBoostClassifier(
            iterations=120 if RUN_MODE == "full" else 55,
            depth=2,
            learning_rate=0.035,
            l2_leaf_reg=12.0,
            loss_function="Logloss",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=1,
        )
    def build_cat_reg(seed=17):
        return CatBoostRegressor(
            iterations=120 if RUN_MODE == "full" else 55,
            depth=2,
            learning_rate=0.035,
            l2_leaf_reg=12.0,
            loss_function="RMSE",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=1,
        )
    for nm, fn, is_reg in [
        ("pos_cat_residual_cdf", build_cat_reg, True),
        ("pos_cat_cls", build_cat_cls, False),
    ]:
        try:
            if is_reg:
                name, trp, tep = pos_regression_candidate(fn, nm)
            else:
                name, trp, tep = pos_classifier_candidate(fn, nm)
            candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)
        except Exception as e:
            print("skip", nm, str(e)[:120])

# All-data horizon candidates: these are deliberately few and regularized.
for nm, builder in [
    ("all_logit_gateblend", lambda j, h, seed=17: build_logit(j, h, seed, C=0.22, balanced=False)),
    ("all_logit_bal_gateblend", lambda j, h, seed=17: build_logit(j, h, seed, C=0.45, balanced=True)),
    ("all_gb_gateblend", build_gb_cls),
]:
    try:
        name, trp, tep = all_horizon_classifier_candidate(builder, nm)
        candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)
    except Exception as e:
        print("skip", nm, str(e)[:120])

if HAS_LGB:
    try:
        name, trp, tep = all_horizon_classifier_candidate(build_lgb_cls, "all_lgb_gateblend")
        candidate_names.append(name); candidate_train.append(trp); candidate_test.append(tep)
    except Exception as e:
        print("skip all_lgb_gateblend", str(e)[:120])

candidate_train = np.stack(candidate_train, axis=0)
candidate_test = np.stack(candidate_test, axis=0)

scores = []
for i, nm in enumerate(candidate_names):
    sc = hybrid_score(candidate_train[i])
    scores.append(sc[0])
    print(f"{i:02d} {nm:28s} score={sc[0]:.6f} c={sc[1]:.6f} wb={sc[2]:.6f} b24={sc[3]:.6f} b48={sc[4]:.6f}")

scores = np.asarray(scores, dtype=float)
n_cand = len(candidate_names)

def blend_arrays(stack, w):
    return enforce_monotone(np.tensordot(w, stack, axes=(0, 0)))

# Score-softmax prior, then random-search non-smooth hybrid objective on OOF.
temperature = 0.0038
prior = np.exp((scores - scores.max()) / temperature)
prior = prior / prior.sum()

rng = np.random.default_rng(20260423)
best_w = prior.copy()
best_sc = hybrid_score(blend_arrays(candidate_train, best_w))[0]

# Include sparse/top-heavy, score-prior-centered, and uniform proposals.
proposal_weights = [prior, np.ones(n_cand) / n_cand]
top_order = np.argsort(-scores)
for k in [2, 3, 5, min(8, n_cand)]:
    w = np.zeros(n_cand)
    w[top_order[:k]] = 1.0 / k
    proposal_weights.append(w)

for w0 in proposal_weights:
    sc = hybrid_score(blend_arrays(candidate_train, w0))[0]
    if sc > best_sc:
        best_sc, best_w = sc, w0.copy()

alpha_center = 1.0 + 35.0 * prior
for _ in range(RANDOM_SEARCH_N):
    if rng.random() < 0.72:
        w = rng.dirichlet(alpha_center)
    else:
        k = int(rng.integers(2, min(7, n_cand) + 1))
        chosen = rng.choice(top_order[:min(n_cand, 10)], size=k, replace=False)
        w = np.zeros(n_cand)
        local = rng.dirichlet(np.ones(k) * 1.4)
        w[chosen] = local
    sc = hybrid_score(blend_arrays(candidate_train, w))[0]
    if sc > best_sc:
        best_sc = sc
        best_w = w.copy()

# Smooth the optimized blend toward the score prior to reduce OOF overfit.
final_w = 0.68 * best_w + 0.32 * prior
final_w = final_w / final_w.sum()
blend_train = blend_arrays(candidate_train, final_w)
blend_test = blend_arrays(candidate_test, final_w)

raw_sc = hybrid_score(blend_train)
print("blend_raw", raw_sc)

# Light OOF calibration. Keep rank signal and only partially map probabilities.
calibrated_test = blend_test.copy()
calibrated_train = blend_train.copy()
cal_alpha = {0: 0.18, 1: 0.28, 2: 0.30}
for j, h in enumerate([12, 24, 48]):
    valid = ~((event_values == 0) & (time_values < h))
    y = ((event_values == 1) & (time_values <= h)).astype(float)
    if len(np.unique(y[valid])) >= 2:
        ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        ir.fit(blend_train[valid, j], y[valid])
        a = cal_alpha[j]
        calibrated_train[:, j] = (1.0 - a) * blend_train[:, j] + a * ir.predict(blend_train[:, j])
        calibrated_test[:, j] = (1.0 - a) * blend_test[:, j] + a * ir.predict(blend_test[:, j])

# Preserve a low but nonzero far-risk floor and force the metric-optimal 72h horizon.
near_test = (test["dist_min_ci_0_5h"].to_numpy(float) < 5000.0)
calibrated_test[:, :3] = np.where(
    near_test[:, None],
    calibrated_test[:, :3],
    np.maximum(calibrated_test[:, :3] * 0.30, floor_test),
)
calibrated_test[:, 3] = 1.0
calibrated_train[:, 3] = 1.0

calibrated_train = enforce_monotone(calibrated_train)
calibrated_test = enforce_monotone(calibrated_test)
print("blend_cal", hybrid_score(calibrated_train))

print("Final blend weights:")
for nm, w, sc in sorted(zip(candidate_names, final_w, scores), key=lambda z: -z[1]):
    if w > 1e-4:
        print(f"  {w:8.5f}  {sc:.6f}  {nm}")

submission = sample[[ID]].copy()
pred_df = pd.DataFrame({ID: test[ID].values})
pred_df[P_COLS] = calibrated_test
submission = submission.merge(pred_df, on=ID, how="left")
submission[P_COLS] = submission[P_COLS].astype(float)
submission[P_COLS] = enforce_monotone(submission[P_COLS].to_numpy(float))
submission["prob_72h"] = 1.0
submission = submission[[ID] + P_COLS]

assert len(submission) == len(sample)
assert submission[ID].is_unique
assert set(submission[ID]) == set(sample[ID])
assert np.isfinite(submission[P_COLS].to_numpy()).all()
assert ((submission[P_COLS] >= 0.0) & (submission[P_COLS] <= 1.0)).all().all()
assert (submission[P_COLS].diff(axis=1).iloc[:, 1:] >= -1e-12).all().all()

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True) if os.path.dirname(OUTPUT_PATH) else None
submission.to_csv(OUTPUT_PATH, index=False)
print("saved", OUTPUT_PATH, submission.shape)
print(submission.describe())


00 physics_5km_timing           score=0.975265 c=0.946920 wb=0.012587 b24=0.026807 b48=0.011361
01 loo_group_km_timing          score=0.970813 c=0.935782 wb=0.014173 b24=0.026284 b48=0.015720
02 pos_ridge_residual_cdf       score=0.965990 c=0.925348 wb=0.016592 b24=0.031798 b48=0.017630
03 pos_bayes_residual_cdf       score=0.973349 c=0.941661 wb=0.013071 b24=0.024949 b48=0.013966
04 pos_huber_residual_cdf       score=0.957999 c=0.914127 wb=0.023198 b24=0.043271 b48=0.025543
05 pos_extratrees_residual_cdf  score=0.973891 c=0.943069 wb=0.012900 b24=0.023461 b48=0.014653
06 pos_logit                    score=0.927588 c=0.921000 wb=0.069589 b24=0.090052 b48=0.106434
07 pos_logit_balanced           score=0.927588 c=0.921000 wb=0.069589 b24=0.090052 b48=0.106434
08 pos_knn                      score=0.968637 c=0.920959 wb=0.010930 b24=0.023678 b48=0.009566
09 pos_extratrees_cls           score=0.969467 c=0.943193 wb=0.019273 b24=0.042610 b48=0.016226
10 pos_gb_stump                 score=0.